#  Kaggle Playground Series S6E2: Heart Disease Prediction

**Competition Goal:** Predict the likelihood of heart disease  
**Evaluation Metric:** Area Under the ROC Curve (AUC-ROC)  
**Link:** https://www.kaggle.com/competitions/playground-series-s6e2

---

##  What We'll Cover

This notebook follows the concepts from our Kaggle sessions:
1. **Data Loading & EDA** 
2. **Data Preprocessing**
3. **Model Training** - (Logistic Regression, Random Forest, Gradient Boosting)
4. **Cross Validation** 
5. **Hyperparameter Tuning** 
6. **Feature Engineering** 
7. **Making Predictions & Submission**

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Settings
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully!")

## 2. Load Data

**Make sure you've downloaded the data from Kaggle and placed it in the same folder as this notebook!**

In [ ]:
# Load datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

## 3. Exploratory Data Analysis (EDA)

### 3.1 First Look at the Data

In [ ]:
# Display first few rows
print("First 5 rows of training data:")
display(train_df.head())

### 3.2 Dataset Information

In [ ]:
# Check data types and missing values
print("\nDataset Info:")
train_df.info()

### 3.3 Statistical Summary

In [ ]:
# Statistical summary
display(train_df.describe())

### 3.4 Check Missing Values

In [ ]:
# Check for missing values
missing = train_df.isnull().sum()
if missing.sum() == 0:
    print("No missing values!")
else:
    print(missing[missing > 0])

### 3.5 Target Variable

In [ ]:
# Target distribution
print("Target Variable Distribution:")
print(train_df['Heart Disease'].value_counts())
print(f"\nClass ratio: {train_df['Heart Disease'].value_counts().values[1]/train_df['Heart Disease'].value_counts().values[0]:.2f}")

## 4. Visualizing the Data

### 4.1 Target Distribution

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_df['Heart Disease'].value_counts().plot(kind='bar', ax=axes[0], color=["#007b33", "#ba1a07"])
axes[0].set_title('Heart Disease Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Absence', 'Presence'], rotation=0)

train_df['Heart Disease'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                                colors=['#007b33', '#ba1a07'], startangle=90)
axes[1].set_title('Heart Disease Percentage', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 4.2 Age Distribution

In [ ]:
# Age by heart disease
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(data=train_df, x='Age', hue='Heart Disease', bins=30, kde=True)
plt.title('Age Distribution', fontweight='bold')

plt.subplot(1, 2, 2)
sns.boxplot(data=train_df, x='Heart Disease', y='Age')
plt.title('Age by Heart Disease', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.3 Correlation Analysis

In [ ]:
# Encode target for correlation
train_encoded = train_df.copy()
train_encoded['Heart Disease'] = (train_encoded['Heart Disease'] == 'Presence').astype(int)

# Correlation heatmap
plt.figure(figsize=(14, 10))
corr = train_encoded.drop('id', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Most correlated with target
print("\nFeatures most correlated with Heart Disease:")
print(corr['Heart Disease'].sort_values(ascending=False)[1:])

### 4.4 Key Features

In [ ]:
# Categorical features vs target
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

pd.crosstab(train_df['Chest pain type'], train_df['Heart Disease']).plot(
    kind='bar', ax=axes[0,0], color=['#007b33', '#ba1a07'])
axes[0,0].set_title('Chest Pain Type', fontweight='bold')
axes[0,0].legend(['Absence', 'Presence'])

pd.crosstab(train_df['Sex'], train_df['Heart Disease']).plot(
    kind='bar', ax=axes[0,1], color=['#007b33', '#ba1a07'])
axes[0,1].set_title('Sex (0=Female, 1=Male)', fontweight='bold')
axes[0,1].legend(['Absence', 'Presence'])

pd.crosstab(train_df['Exercise angina'], train_df['Heart Disease']).plot(
    kind='bar', ax=axes[1,0], color=['#007b33', '#ba1a07'])
axes[1,0].set_title('Exercise Angina', fontweight='bold')
axes[1,0].legend(['Absence', 'Presence'])

pd.crosstab(train_df['Thallium'], train_df['Heart Disease']).plot(
    kind='bar', ax=axes[1,1], color=['#007b33', '#ba1a07'])
axes[1,1].set_title('Thallium', fontweight='bold')
axes[1,1].legend(['Absence', 'Presence'])

plt.tight_layout()
plt.show()

## 5. Data Preprocessing

### 5.1 Prepare Data

In [ ]:
# Encode target: Presence -> 1, Absence -> 0
train_df['Heart Disease'] = (train_df['Heart Disease'] == 'Presence').astype(int)

# Separate features and target
X = train_df.drop(['id', 'Heart Disease'], axis=1)
y = train_df['Heart Disease']
X_test = test_df.drop('id', axis=1)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Test shape: {X_test.shape}")

### 5.2 Train-Validation Split

In [ ]:
# Split data (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

## 6. Model Training 

We'll train three models:
- **Logistic Regression** - Our baseline
- **Random Forest** - Tree-based ensemble
- **Gradient Boosting** - Boosting algorithm

### 6.1 Logistic Regression

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train model
print("Logistic Regression training in progress")
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_train_pred = lr_model.predict_proba(X_train_scaled)[:, 1]
lr_val_pred = lr_model.predict_proba(X_val_scaled)[:, 1]

# Scores
lr_train_auc = roc_auc_score(y_train, lr_train_pred)
lr_val_auc = roc_auc_score(y_val, lr_val_pred)

print(f"Training AUC: {lr_train_auc:.4f}")
print(f"Validation AUC: {lr_val_auc:.4f}")

### 6.2 Random Forest

In [ ]:
# Train Random Forest
print("Random Forest training in progress")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predictions
rf_train_pred = rf_model.predict_proba(X_train)[:, 1]
rf_val_pred = rf_model.predict_proba(X_val)[:, 1]

# Scores
rf_train_auc = roc_auc_score(y_train, rf_train_pred)
rf_val_auc = roc_auc_score(y_val, rf_val_pred)

print(f"Training AUC: {rf_train_auc:.4f}")
print(f"Validation AUC: {rf_val_auc:.4f}")

### 6.3 Gradient Boosting

In [ ]:
# Train Gradient Boosting
print("Training Gradient Boosting")
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=RANDOM_STATE
)
gb_model.fit(X_train, y_train)

# Predictions  
gb_train_pred = gb_model.predict_proba(X_train)[:, 1]
gb_val_pred = gb_model.predict_proba(X_val)[:, 1]

# Scores
gb_train_auc = roc_auc_score(y_train, gb_train_pred)
gb_val_auc = roc_auc_score(y_val, gb_val_pred)

print(f"Training AUC: {gb_train_auc:.4f}")
print(f"Validation AUC: {gb_val_auc:.4f}")

### 6.4 Compare Models

In [ ]:
# Create comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Training AUC': [lr_train_auc, rf_train_auc, gb_train_auc],
    'Validation AUC': [lr_val_auc, rf_val_auc, gb_val_auc]
})
results['Overfit Gap'] = results['Training AUC'] - results['Validation AUC']

print("\nModel Comparison:")
display(results.round(4))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results))
width = 0.35

ax.bar(x - width/2, results['Training AUC'], width, label='Training', alpha=0.8)
ax.bar(x + width/2, results['Validation AUC'], width, label='Validation', alpha=0.8)

ax.set_xlabel('Model', fontweight='bold')
ax.set_ylabel('AUC Score', fontweight='bold')
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. ROC Curves

In [ ]:
# Calculate ROC curves
lr_fpr, lr_tpr, _ = roc_curve(y_val, lr_val_pred)
rf_fpr, rf_tpr, _ = roc_curve(y_val, rf_val_pred)
gb_fpr, gb_tpr, _ = roc_curve(y_val, gb_val_pred)

# Plot
plt.figure(figsize=(10, 8))
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {lr_val_auc:.4f})', linewidth=2)
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {rf_val_auc:.4f})', linewidth=2)
plt.plot(gb_fpr, gb_tpr, label=f'Gradient Boosting (AUC = {gb_val_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')

plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('ROC Curves', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Cross-Validation

In [ ]:
# 5-fold CV on Random Forest
print("Performing 5-Fold Cross-Validation...\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"CV Scores: {cv_scores}")
print(f"Mean AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Visualize
plt.figure(figsize=(10, 6))
plt.bar(range(1, 6), cv_scores, alpha=0.8)
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold', fontweight='bold')
plt.ylabel('AUC Score', fontweight='bold')
plt.title('Cross-Validation Scores', fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 9. Hyperparameter Tuning

In [ ]:
# Test different configurations
print("Testing hyperparameters...\n")

configs = [
    {'n_estimators': 100, 'max_depth': 10, 'name': 'Config 1 (Default)'},
    {'n_estimators': 150, 'max_depth': 12, 'name': 'Config 2'},
    {'n_estimators': 200, 'max_depth': 8, 'name': 'Config 3'},
]

tuning_results = []
for config in configs:
    name = config.pop('name')
    model = RandomForestClassifier(**config, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    
    val_pred = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_pred)
    
    tuning_results.append({'Config': name, 'Validation AUC': val_auc})
    print(f"{name}: {val_auc:.4f}")

tuning_df = pd.DataFrame(tuning_results)
print("\nBest config:", tuning_df.loc[tuning_df['Validation AUC'].idxmax(), 'Config'])

## 10. Feature Engineering

Now let's create new features to improve performance!

### 10.1 Create Features

In [ ]:
def engineer_features(df):
    """Create new features"""
    df = df.copy()
    
    # Age groups (binning)
    df['Age_Group'] = pd.cut(df['Age'], bins=[0, 45, 55, 65, 100], labels=[0,1,2,3]).astype(int)
    
    # BP categories (extended range to catch all values)
    df['BP_Category'] = pd.cut(df['BP'], bins=[0, 120, 140, 250], labels=[0,1,2]).astype(int)
    
    # Cholesterol categories (extended range to catch all values)
    df['Chol_Category'] = pd.cut(df['Cholesterol'], bins=[0, 200, 240, 600], labels=[0,1,2]).astype(int)
    
    # Interaction features
    df['Age_BP'] = df['Age'] * df['BP']
    df['Age_Chol'] = df['Age'] * df['Cholesterol']
    df['ChestPain_Angina'] = df['Chest pain type'] * df['Exercise angina']
    
    # Risk score (domain knowledge)
    df['Risk_Score'] = (
        df['Age']/100 + df['BP']/200 + 
        df['Cholesterol']/400 + df['Exercise angina']*2 + df['ST depression']
    )
    
    return df

# Apply
X_train_fe = engineer_features(X_train)
X_val_fe = engineer_features(X_val)
X_test_fe = engineer_features(X_test)

print(f"Original features: {X_train.shape[1]}")
print(f"After engineering: {X_train_fe.shape[1]}")
print(f"New features: {X_train_fe.shape[1] - X_train.shape[1]}")

### 10.2 Train with New Features

In [ ]:
# Train with best config and new features
print("Training with engineered features...")
rf_fe_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_fe_model.fit(X_train_fe, y_train)

rf_fe_val_pred = rf_fe_model.predict_proba(X_val_fe)[:, 1]
rf_fe_val_auc = roc_auc_score(y_val, rf_fe_val_pred)

print(f"\nValidation AUC: {rf_fe_val_auc:.4f}")
print(f"Improvement: {rf_fe_val_auc - rf_val_auc:+.4f}")

### 10.3 Feature Importance

In [ ]:
# Get feature importance
importance = pd.DataFrame({
    'Feature': X_train_fe.columns,
    'Importance': rf_fe_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15
plt.figure(figsize=(10, 8))
top15 = importance.head(15)
sns.barplot(data=top15, y='Feature', x='Importance')
plt.title('Top 15 Most Important Features', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 10 Features:")
display(importance.head(10))

## 11. Generate Predictions

In [ ]:
# Make predictions on test set
print("Generating test predictions...")
test_pred = rf_fe_model.predict_proba(X_test_fe)[:, 1]

print(f"\nPredictions: {len(test_pred)}")
print(f"Min: {test_pred.min():.4f}")
print(f"Max: {test_pred.max():.4f}")
print(f"Mean: {test_pred.mean():.4f}")

# Visualize distribution
plt.figure(figsize=(10, 6))
plt.hist(test_pred, bins=50, edgecolor='black')
plt.xlabel('Predicted Probability', fontweight='bold')
plt.ylabel('Count', fontweight='bold')
plt.title('Test Predictions Distribution', fontweight='bold')
plt.axvline(test_pred.mean(), color='red', linestyle='--', label=f'Mean: {test_pred.mean():.4f}')
plt.legend()
plt.show()

## 12. Create Submission

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'id': test_df['id'],
    'Heart Disease': test_pred
})

submission.to_csv('submission.csv', index=False)

print("Submission created\n")
print("Preview:")
display(submission.head(10))
print(f"\nTotal rows: {len(submission)}")
print("\nFile saved as 'submission.csv'")


## 13. Next Steps 

### Ways to Improve:

**Advanced Models:**
- Try XGBoost, LightGBM, CatBoost
- Ensemble multiple models
- Stack models together

**Better Hyperparameter Tuning:**
- Use GridSearchCV or RandomizedSearchCV
- Try more hyperparameter combinations
- Optimize different model types

**More Features:**
- Create polynomial features
- Try different binning strategies
- Research medical risk factors
- Feature selection methods

**Better Validation:**
- Use more CV folds
- Try RepeatedStratifiedKFold
- Track overfitting carefully

**Data:**
- Use the original dataset (https://www.kaggle.com/datasets/neurocipher/heartdisease/data)
- Try data augmentation
- Handle outliers


Good luck! 